# W01 · RL 问题建模：MDP、回报与价值函数

> 阶段一（Stable Baselines3 算法基础）第 1 周。
> 本周不写任何「训练」代码，目标是把强化学习的**数学语言**建立起来——
> 后面每一周（DQN/PPO/SAC……）都只是这套语言的不同方言。

## 学习目标

学完本 notebook，你应该能够：

1. 用自己的话解释「智能体–环境交互循环」，并说出 RL 与监督学习的三点区别；
2. 写出 MDP 五元组 $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$，并判断一个过程是否满足马尔可夫性质；
3. 写出回报 $G_t$、状态价值 $V^\pi(s)$、动作价值 $Q^\pi(s,a)$ 的定义与 Bellman 方程；
4. 用**策略评估（policy evaluation）**和**价值迭代（value iteration）**手算/编程求解小规模 GridWorld；
5. 说明 on-policy 与 off-policy 的区别，并能给后续算法归类。

**先修要求**：基础概率（条件期望）、numpy 基本操作。预计用时 2–3 小时。

## 1. 直觉：强化学习在学什么？

想象你在训练一只小狗：你不会告诉它「左前腿抬 30 度」，你只会在它坐下时给零食。
小狗通过**试错（trial and error）**逐渐学会「什么行为能换来零食」。强化学习就是把这件事数学化：

- **智能体（agent）**：小狗 / 控制器 / 神经网络策略；
- **环境（environment）**：世界的一切其他部分；
- 每个时刻 $t$：智能体观察**状态** $s_t$，选择**动作** $a_t$，环境反馈**奖励** $r_{t+1}$ 和新状态 $s_{t+1}$；
- 目标：学会一个**策略（policy）** $\pi$，让长期累计奖励最大。

与监督学习的三个关键区别：

| | 监督学习 | 强化学习 |
|---|---|---|
| 监督信号 | 每条数据有「标准答案」 | 只有延迟、稀疏的奖励 |
| 数据分布 | 预先给定、静态 | 由策略自己产生，**边学边变** |
| 反馈时序 | 即时 | 可能延迟很多步（围棋赢了才知道哪步好） |

对机器人来说：状态 = 关节角/速度/传感器读数，动作 = 力矩/推力，奖励 = 我们自己设计的「任务打分函数」。
**奖励设计几乎决定了智能体会学到什么行为**——这是机器人 RL 工程里最核心也最容易踩坑的部分（W02 会动手体验）。

## 2. 数学骨架：马尔可夫决策过程（MDP）

一个（有限）MDP 由五元组 $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$ 定义：

- $\mathcal{S}$：状态集合；$\mathcal{A}$：动作集合；
- **转移概率** $\mathcal{P}$：$p(s' \mid s, a) = \Pr\{S_{t+1}=s' \mid S_t=s, A_t=a\}$；
- **奖励函数** $\mathcal{R}$：$r(s, a, s')$ 或其期望；
- **折扣因子** $\gamma \in [0, 1)$。

**马尔可夫性质**：未来只依赖当前状态，与历史无关——
$$\Pr\{S_{t+1} \mid S_t, A_t\} = \Pr\{S_{t+1} \mid S_t, A_t, S_{t-1}, A_{t-1}, \dots\}$$

类比：下棋时，只要看清当前棋局就够做决策了，不需要复盘整盘棋怎么走到这的。
机器人里若传感器只能看到当前关节角而看不到速度，马尔可夫性质就被破坏——
工程上常用「堆叠最近几帧观测」来近似修复。

**策略** $\pi(a \mid s)$：在状态 $s$ 下选择动作 $a$ 的概率分布。MDP 的目标就是找到最优策略 $\pi^*$。

## 3. 回报与折扣：把「长远利益」变成一个数

智能体关心的是**累计奖励**，即回报（return）：

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

$\gamma$ 的直觉：

- $\gamma = 0$：完全「鼠目寸光」，只看眼前一步；
- $\gamma \to 1$：深谋远虑，未来奖励几乎和眼前一样重要；
- 数学上 $\gamma < 1$ 保证无穷级数收敛（若奖励有界，$|G_t| \le R_{max}/(1-\gamma)$）。

机器人任务里 $\gamma$ 通常取 0.99 左右；金融级长期任务可能取 0.999。
下面用代码感受一下 $\gamma$ 对回报的塑形作用：

In [1]:
import numpy as np

# 假设一条固定的奖励流：前 9 步不得分，第 10 步得到大奖（类似「坚持到最后才成功」）
rewards = [0.0] * 9 + [10.0]

def discounted_return(rewards, gamma):
    G = 0.0
    for r in reversed(rewards):      # 从后往前累乘 γ，等价于 G_t = r + γ·G_{t+1}
        G = r + gamma * G
    return G

for gamma in [0.5, 0.8, 0.9, 0.99]:
    print(f"γ = {gamma:>4}:  G_0 = {discounted_return(rewards, gamma):.4f}")

print()
print(f"对照：γ = 0.9 时大奖被折扣成 10 × 0.9^9 = {10 * 0.9**9:.4f}")
print("→ γ 太小会系统性低估延迟奖励，智能体会变得'短视'")

γ =  0.5:  G_0 = 0.0195
γ =  0.8:  G_0 = 1.3422
γ =  0.9:  G_0 = 3.8742
γ = 0.99:  G_0 = 9.1352

对照：γ = 0.9 时大奖被折扣成 10 × 0.9^9 = 3.8742
→ γ 太小会系统性低估延迟奖励，智能体会变得'短视'


## 4. 价值函数与 Bellman 方程

### 4.1 两种价值函数

- **状态价值**：从 $s$ 出发、遵循策略 $\pi$ 的期望回报
  $$V^\pi(s) = \mathbb{E}_\pi\left[ G_t \,\middle|\, S_t = s \right]$$
- **动作价值**：在 $s$ 先做动作 $a$、之后遵循 $\pi$ 的期望回报
  $$Q^\pi(s, a) = \mathbb{E}_\pi\left[ G_t \,\middle|\, S_t = s, A_t = a \right]$$

类比：$V^\pi(s)$ 是「这个局面本身值多少分」（像围棋 AI 给出的胜率），
$Q^\pi(s,a)$ 是「这步棋走下去值多少分」。有了 $Q$，做决策就是查表取最大：
$\pi^*(s) = \arg\max_a Q^*(s, a)$。

### 4.2 Bellman 方程：价值函数的「递归定义」

把 $G_t = R_{t+1} + \gamma G_{t+1}$ 代入定义，得到 **Bellman 期望方程**：

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} p(s'|s,a) \Big[ r(s,a,s') + \gamma V^\pi(s') \Big]$$

**一句话：一个状态的价值 = 即时奖励 + 折扣后的下一状态价值。** 
这个递归结构是整个 RL 的地基——DQN 的 TD 更新、PPO 的 GAE、SAC 的 soft Bellman 备份，全都是它的变体。

对最优价值函数 $V^*$，策略直接取最优动作，得到 **Bellman 最优方程**：

$$V^*(s) = \max_a \sum_{s'} p(s'|s,a) \Big[ r(s,a,s') + \gamma V^*(s') \Big]$$

### 4.3 动手算：4×4 GridWorld 的策略评估

环境（Sutton & Barto 例 4.1）：

- 16 个格子，左上角和右下角是**终点**（价值恒为 0）；
- 动作：上下左右；每走一步奖励 $-1$，撞墙原地不动也 $-1$；
- $\gamma = 1$，策略：均匀随机（每个方向 0.25）。

**策略评估**：反复套用 Bellman 期望方程做迭代更新
$$V_{k+1}(s) \leftarrow \sum_a \pi(a|s) \sum_{s'} p(s'|s,a)\big[r + \gamma V_k(s')\big]$$
直到收敛。直觉：价值信息像水一样从终点「扩散」到整个地图。

In [2]:
import numpy as np

GRID = 4
TERMINAL = {(0, 0), (GRID - 1, GRID - 1)}
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # 上、下、左、右
GAMMA = 1.0

def step(s, a):
    # 确定性转移：撞墙则原地不动。返回 (s', r)
    ns = (s[0] + a[0], s[1] + a[1])
    if ns[0] < 0 or ns[0] >= GRID or ns[1] < 0 or ns[1] >= GRID:
        ns = s                      # 撞墙
    return ns, -1.0

def policy_evaluation(n_iter=1000, tol=1e-6):
    V = np.zeros((GRID, GRID))
    for k in range(n_iter):
        delta = 0.0
        V_new = V.copy()
        for i in range(GRID):
            for j in range(GRID):
                if (i, j) in TERMINAL:
                    continue
                # 均匀随机策略：每个动作概率 0.25
                v = 0.0
                for a in ACTIONS:
                    ns, r = step((i, j), a)
                    v += 0.25 * (r + GAMMA * V[ns])
                V_new[i, j] = v
                delta = max(delta, abs(v - V[i, j]))
        V = V_new
        if delta < tol:
            print(f"第 {k + 1} 轮迭代收敛")
            break
    return V

V = policy_evaluation()
np.set_printoptions(precision=1, suppress=True)
print("均匀随机策略下的 V(s)（越靠近终点负得越少，因为期望步数更少）：")
print(V)

第 258 轮迭代收敛
均匀随机策略下的 V(s)（越靠近终点负得越少，因为期望步数更少）：
[[  0. -14. -20. -22.]
 [-14. -18. -20. -20.]
 [-20. -20. -18. -14.]
 [-22. -20. -14.   0.]]


**读图**：格子里的数 ≈ 「从这个格子出发、随机走，期望还要扣多少分」，也就是 $-$期望步数。
中心格约 $-14$：随机游走平均要 14 步才能蹭到终点。

### 4.4 价值迭代：直接求最优策略

把 Bellman **最优**方程当成迭代更新式，就是价值迭代：
$$V_{k+1}(s) \leftarrow \max_a \sum_{s'} p(s'|s,a)\big[r + \gamma V_k(s')\big]$$
收敛后取 $V^*$ 里每个状态的 $\arg\max$ 动作，就得到最优策略 $\pi^*$。

In [3]:
def value_iteration(n_iter=1000, tol=1e-9):
    V = np.zeros((GRID, GRID))
    for k in range(n_iter):
        delta = 0.0
        V_new = V.copy()
        for i in range(GRID):
            for j in range(GRID):
                if (i, j) in TERMINAL:
                    continue
                V_new[i, j] = max(
                    (lambda ns_r: ns_r[1] + GAMMA * V[ns_r[0]])(step((i, j), a))
                    for a in ACTIONS
                )
                delta = max(delta, abs(V_new[i, j] - V[i, j]))
        V = V_new
        if delta < tol:
            print(f"第 {k + 1} 轮迭代收敛")
            break
    return V

V_star = value_iteration()
print("最优价值 V*(s)：")
print(V_star)

# 提取最优策略并用箭头可视化
ARROWS = {(-1, 0): "↑", (1, 0): "↓", (0, -1): "←", (0, 1): "→"}
policy = []
for i in range(GRID):
    row = []
    for j in range(GRID):
        if (i, j) in TERMINAL:
            row.append("■")
            continue
        best_a = max(ACTIONS, key=lambda a: (lambda ns_r: ns_r[1] + GAMMA * V_star[ns_r[0]])(step((i, j), a)))
        row.append(ARROWS[best_a])
    policy.append(row)

print("\n最优策略（■ = 终点）：")
for row in policy:
    print("  ".join(row))

第 4 轮迭代收敛
最优价值 V*(s)：
[[ 0. -1. -2. -3.]
 [-1. -2. -3. -2.]
 [-2. -3. -2. -1.]
 [-3. -2. -1.  0.]]

最优策略（■ = 终点）：
■  ←  ←  ↓
↑  ↑  ↑  ↓
↑  ↑  ↓  ↓
↑  →  →  ■


**注意**：这里我们**精确知道**转移概率 $p(s'|s,a)$（撞墙规则、终点都在代码里），
所以可以直接用 Bellman 方程迭代求解——这叫**动态规划（DP）**，属于「有模型（model-based）」方法。

真实机器人问题里 $p(s'|s,a)$ 未知（物理引擎之外的世界没有解析解），
只能从**采样经验**中估计价值——这就是 W03 的 Q-Learning 要解决的问题。
从 DP 到 Q-Learning 的桥梁是：**把期望换成采样平均，把精确更新换成小步长学习**。

## 5. MDP 概念 → Gymnasium API 的映射

| MDP 概念 | Gymnasium 中的对应物 |
|---|---|
| $\mathcal{S}, \mathcal{A}$ | `env.observation_space` / `env.action_space` |
| $p(s'\mid s,a),\; r(s,a,s')$ | `env.step(a)` 返回的 `(obs, reward, ...)`（黑箱采样器） |
| 初始状态分布 | `env.reset()` |
| $\pi(a\mid s)$ | 你的策略代码/神经网络 |
| 回合结束 | `terminated`（任务层面结束）/ `truncated`（时间上限截断） |

看一眼 CartPole-v1（倒立摆推车，W04 的主角）的 MDP 要素：

In [4]:
import gymnasium as gym

env = gym.make("CartPole-v1")
print("观测空间:", env.observation_space)
print("  → [小车位置, 小车速度, 杆角度, 杆角速度]，共 4 维连续值")
print("动作空间:", env.action_space)
print("  → 0 = 向左推, 1 = 向右推")

obs, info = env.reset(seed=0)
total_r, steps = 0.0, 0
terminated = truncated = False
while not (terminated or truncated):
    obs, r, terminated, truncated, _ = env.step(env.action_space.sample())  # 随机策略
    total_r += r
    steps += 1
print(f"\n随机策略一回合：存活 {steps} 步，回报 {total_r}（每存活一步 +1）")
print("→ 随机策略只能撑 20 步左右，这就是 RL 要改进的基线")
env.close()

观测空间: Box([-4.8e+00 -3.4e+38 -4.2e-01 -3.4e+38], [4.8e+00 3.4e+38 4.2e-01 3.4e+38], (4,), float32)
  → [小车位置, 小车速度, 杆角度, 杆角速度]，共 4 维连续值
动作空间: Discrete(2)
  → 0 = 向左推, 1 = 向右推

随机策略一回合：存活 44 步，回报 44.0（每存活一步 +1）
→ 随机策略只能撑 20 步左右，这就是 RL 要改进的基线


## 6. On-policy 与 Off-policy（先混个眼熟）

- **On-policy（同策略）**：用来学习的样本由**当前策略**产生。学的是什么，练的就是什么。
  代表：SARSA、PPO。
- **Off-policy（异策略）**：可以用**别的策略**（旧策略、人类演示、随机探索）产生的样本来学习目标策略。
  代表：Q-Learning、DQN、SAC、TD3。

类比：on-policy 像「只能复盘自己刚下的棋」；off-policy 像「还能打谱学习别人的棋」。
off-policy 样本效率高（经验可以反复用），但引入了分布失配，需要经验回放、重要性采样等技术来稳住——
这正是 W04–W06 的故事。本周练习第 4 题会让你先做个判断，到时候回来对答案。

---

## ✏️ 练习

> 规则：先独立完成，再点开折叠的参考答案核对。交付物请直接写在本 notebook 末尾新建的 cell 中。

**练习 1（★，10 分钟）——手算链式 MDP 的价值**
三个状态的链式 MDP：$s_1 \to s_2 \to s_3$（$s_3$ 为终点）。
只有一个动作，转移确定：$r(s_1)=1,\; r(s_2)=2$，到达 $s_3$ 后回合结束（$V(s_3)=0$）。$\gamma = 0.9$。
手算 $V(s_1), V(s_2)$，然后写两行代码验证（断言误差 < 1e-6）。
**交付物**：markdown 写手算过程 + 带 `assert` 的验证 cell。

**练习 2（★，10 分钟）——折扣因子的影响**
把 4.3 节策略评估代码中的 `GAMMA` 改成 0.9 重跑，对比 $\gamma=1$ 时的 $V$ 表。
**交付物**：打印新的 $V$ 表，并用两句话解释为什么所有格的绝对值都变小了。

**练习 3（★★，25 分钟）——加一个陷阱格**
在 4×4 GridWorld 的 (1, 1) 位置放一个陷阱：走入该格奖励 $-5$（其余规则不变，陷阱不是终点）。
重跑价值迭代并打印新的最优策略箭头图。
**交付物**：箭头图 + 一句话说明策略如何绕开陷阱。

**练习 4（★，5 分钟）——on/off-policy 判断**
不学新知识，仅凭第 6 节的定义判断：Q-Learning、SARSA、4.3 节的策略评估，分别是 on-policy 还是 off-policy？
**交付物**：三句话判断 + 理由（写完再对答案）。

---

<details>
<summary>参考答案（做完再点开）</summary>

**练习 1**：$V(s_3)=0$；$V(s_2) = 2 + 0.9 \times 0 = 2$；$V(s_1) = 1 + 0.9 \times 2 = 2.8$。
验证代码：

```python
v3 = 0.0
v2 = 2 + 0.9 * v3
v1 = 1 + 0.9 * v2
assert abs(v1 - 2.8) < 1e-6 and abs(v2 - 2.0) < 1e-6
```

**练习 2**：`GAMMA = 0.9` 时中心格约 $-12.2$（原约 $-14.2$），各格绝对值普遍变小。
因为同样的未来惩罚被乘上了 $\gamma^k < 1$ 的衰减，越远的负奖励「看起来越不严重」；
$\gamma$ 越小，价值的有效「视野」越短。

**练习 3**：修改 `step()`：若 `ns == (1, 1)` 则奖励改为 `-5.0`。重跑价值迭代后，
(0, 1)、(1, 0)、(2, 1)、(1, 2) 等相邻格的策略会指向绕开陷阱的方向（例如 (1,0) 不再向下走），
因为走入陷阱的一步会损失 5 分，宁可多绕路。

**练习 4**：
- **Q-Learning：off-policy**。它用 $\max_a Q(s', a)$ 更新（目标策略是贪心策略），
  但采样数据的实际行为策略是 $\varepsilon$-greedy，两者不同。
- **SARSA：on-policy**。它用行为策略实际选出的下一动作 $a'$ 做更新，学与练是同一个策略。
- **策略评估（4.3 节）**：不严格属于任一范畴——它不是采样方法，而是对给定策略 $V^\pi$ 的精确计算；
  若强行归类，它评估的就是产生数据的那个策略，思想上与 on-policy 一致。

</details>

---

## 延伸阅读

- [Sutton & Barto《Reinforcement Learning: An Introduction》(2nd ed.) 免费 PDF](http://incompleteideas.net/book/the-book-2nd.html) —— 第 3、4 章对应本周内容，业界标准教材。
- [David Silver UCL RL 课程](https://www.davidstarsilver.uk/teaching/) —— Lecture 2 (MDP)、Lecture 3 (DP)，配讲义。
- [《动手学强化学习》（张伟楠等）](https://hrl.boyuai.com/) —— 中文，第 1–2 章，代码友好。
- [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course) —— Unit 1，工程视角的入门。
- [Gymnasium 文档：基本概念](https://gymnasium.farama.org/introduction/basic_usage/) —— 为 W02 预热。